# Nhóm 2, Notebook_D (Sinh viên B, giai đoạn 2): chi nhãn thế nào cho đúng

**Đề tài**: Khoảng dự báo có cam kết bao phủ cho dung lượng pin xe điện, khi đổi xe và khi đổi thời kỳ

**Vai trò**: Notebook B kết thúc ở một câu để ngỏ. Nó chứng minh rằng khi gặp xe lạ, cam kết bao
phủ 90 % bị hỏng, và cách duy nhất sửa được là **có nhãn của chính chiếc xe đó**. Nhưng nó dừng
ở đó: chia đều $k$ phiên cho mỗi xe, giữ mô hình cố định, hiệu chuẩn chung cho cả đội. Notebook
này lấy ba giả định đó ra và kiểm từng cái, đúng ba hướng R1, R2, R3 trong Bảng 5.

**Đầu vào**: `ev_data/data/sessions.csv`, `results/dac_trung_chon.csv` do Notebook C bàn giao.
**Đầu ra**: `results/mondrian.csv`, `results/ngan_sach_thich_nghi.csv`,
`results/huan_luyen_hay_hieu_chuan.csv` và các hình trong `fig/`.
**Thời gian chạy đo được**: khoảng 6 phút trên 2 lõi CPU. Đặt `NHANH = True` để thử trước.

## Ba câu hỏi, ba bước

| Hướng | Notebook B giả định gì | Notebook D hỏi gì | Bước |
|---|---|---|---|
| R1 | một phân vị hiệu chuẩn chung cho mọi phiên | chia phiên thành nhóm con rồi hiệu chuẩn riêng từng nhóm thì sao | Bước 3 |
| R2 | mỗi xe được đúng $k$ phiên có nhãn như nhau | cùng tổng ngân sách, dồn nhãn vào xe đang hụt nhiều nhất thì sao | Bước 4 |
| R3 | phiên có nhãn luôn dùng để hiệu chuẩn | dùng nó để huấn luyện lại thì tốt hơn hay tệ hơn | Bước 5 |

## Vì sao 

Một phiên sạc có nhãn tốn công: phải biết dung lượng thật của pack tại thời điểm đó. Với một đội
xe thương mại, ngân sách nhãn là ràng buộc thật, không phải bài tập. Notebook B nói *"cần
$k$ phiên"*. Người vận hành đội xe sẽ hỏi lại ba câu: chia thế nào, cho xe nào, và dùng vào
việc gì. Ba câu đó là nội dung của notebook này.

**Một kết quả âm cũng là kết quả.** Nếu Mondrian không cải thiện gì, đó vẫn là một dòng đáng
viết, vì nó thu hẹp không gian giải pháp. Đừng sửa thí nghiệm cho tới khi nó ra số đẹp.

## Vừa làm vừa hỏi AI và ghi Audit Log

Notebook này có hai cái bẫy mà AI hay trả lời sai:

> **Bẫy 1.** Hỏi *"làm Mondrian conformal giúp tôi"*, nhiều câu trả lời sẽ chia nhóm bằng một
> đại lượng phụ thuộc nhãn, ví dụ chia theo dung lượng thật. Như vậy là gian lận: lúc dự báo
> bạn chưa biết nhãn nên không biết điểm mới thuộc nhóm nào. Nhóm phải xác định được **chỉ từ
> đầu vào**.
>
> **Bẫy 2.** Hỏi *"so sánh phân bổ thích nghi với chia đều"*, nhiều câu trả lời sẽ dùng chính
> tập kiểm tra để quyết định chia nhãn cho xe nào. Đó là rò rỉ. Phải chia ngân sách làm hai
> chặng: chặng thăm dò quyết định, chặng sau mới chi, và đánh giá trên phần chưa đụng tới.

| Ngày, bước | Prompt (rút gọn) | AI trả lời gì | Tôi kiểm tra hoặc sửa gì | Dùng vào đâu |
|---|---|---|---|---|
| 04/10, Bước 3 | làm Mondrian conformal | chia nhóm theo dung lượng thật | đổi sang gom cụm trên đặc trưng đầu vào, khớp cụm trên nguồn rồi áp cho đích | Bước 3 |
| 04/10, Bước 4 | phân bổ ngân sách thích nghi | dùng bao phủ trên tập kiểm tra để chọn xe | tách hai chặng thăm dò và chi, đánh giá trên phần còn lại | Bước 4 |

## Bước 0: nạp dữ liệu và bộ đặc trưng đã chọn

Bộ đặc trưng lấy từ `results/dac_trung_chon.csv` của Notebook C. Nếu chưa có file đó, notebook
vẫn chạy được với cả 167 cột, chỉ chậm hơn và có cảnh báo, vì như vậy là bỏ qua bàn giao.

In [ ]:
import json, pathlib, hashlib, warnings, time
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 9, 'axes.grid': True, 'grid.alpha': .25})
TEAL, RUST, SAND, SLATE = '#0F3D3E', '#B5533C', '#C9A227', '#4B6A8A'

SEED = 42
ALPHA = 0.10
NOM = 1 - ALPHA
NHANH = False                    # True: 6 xe thay vì 20
N_CAL_NGUON = 2000               # cỡ tập hiệu chuẩn lấy từ các xe nguồn
np.random.seed(SEED)

DATA = pathlib.Path('ev_data/data')
RES = pathlib.Path('results'); RES.mkdir(exist_ok=True)
FIG = pathlib.Path('fig'); FIG.mkdir(exist_ok=True)

d = pd.read_csv(DATA / 'sessions.csv', parse_dates=['t_start']).dropna(subset=['capacity'])
d = d.reset_index(drop=True)
META = ['vehicle', 'session', 't_start', 'n_rows', 'ah', 'capacity', 'bms_cap']
CAM = ['n_rows', 'ah', 'bms_cap']
FEATS_ALL = [c for c in d.columns if c not in META]

f_chon = RES / 'dac_trung_chon.csv'
if f_chon.exists():
    FEATS = pd.read_csv(f_chon, header=None)[0].tolist()
    print(f'dùng {len(FEATS)} đặc trưng do Notebook C chọn')
else:
    FEATS = FEATS_ALL
    print(f'CẢNH BÁO: chưa có {f_chon}, tạm dùng cả {len(FEATS)} cột. '
          'Hãy chạy Notebook C trước rồi chạy lại notebook này.')
assert set(FEATS) <= set(FEATS_ALL) and not (set(FEATS) & set(CAM))

XE = [int(v) for v in d.vehicle.unique()]
if NHANH: XE = XE[::max(1, len(XE)//6)][:6]
print(f'{len(d):,} phiên, {len(XE)} xe dùng trong notebook này')
print(f'dung lượng: trung vị {d.capacity.median():.1f} Ah, '
      f'độ lệch chuẩn {d.capacity.std():.1f} Ah')

## Bước 1: công cụ conformal, giống Notebook B

Nhắc lại chỗ duy nhất dễ sai: mức phân vị là $\lceil (n+1)(1-\alpha)\rceil/n$ lấy với
`method='higher'`, **không phải** `np.quantile(s, 0.9)`. Cả notebook này sống ở vùng $n$ nhỏ,
nơi hai cách lệch hẳn nhau.

In [ ]:
from xgboost import XGBRegressor
CAP_LO, CAP_HI = 60.0, 220.0

def q_level(n, a=ALPHA):
    # Mức phân vị hữu hạn mẫu, kẹp về 1,0 khi n quá nhỏ.
    return min(1.0, np.ceil((n + 1) * (1 - a)) / n)

def q_conformal(res, a=ALPHA):
    res = np.abs(np.asarray(res, dtype=float))
    if len(res) == 0: return np.inf
    return float(np.quantile(res, q_level(len(res), a), method='higher'))

def bao_phu_tu_res(res_te, q):
    # Với điểm số sai số tuyệt đối, khoảng là p +- q nên phủ khi |res| <= q.
    res_te = np.abs(np.asarray(res_te, dtype=float))
    return float((res_te <= q).mean()), float(2 * q)

def tao_xgb(seed=SEED):
    return XGBRegressor(n_estimators=600, max_depth=6, learning_rate=.05, subsample=.8,
                        colsample_bytree=.7, min_child_weight=5, reg_lambda=2.,
                        tree_method='hist', n_jobs=2, random_state=seed)

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b))**2)))

for n in (5, 10, 25, 50, 2000):
    print(f'n={n:5d} -> mức phân vị {q_level(n):.4f}'
          + ('   <- đã kẹp về 1,0' if q_level(n) >= 1.0 else ''))

## Bước 2: dựng nền, giữ lại từng xe một lần

Với mỗi xe đích $v$:

| Tập | Lấy từ đâu | Dùng làm gì |
|---|---|---|
| huấn luyện | 19 xe còn lại, 80 % | khớp XGBoost |
| hiệu chuẩn nguồn | 19 xe còn lại, 20 %, lấy tối đa 2000 phiên | phân vị conformal khi chưa có nhãn của xe đích |
| đích | toàn bộ phiên của xe $v$ | nơi đo bao phủ, và là nơi ngân sách nhãn được chi |

Khớp **một lần** cho mỗi xe rồi giữ lại dự báo. Mọi thí nghiệm ở Bước 3 và Bước 4 chỉ là phép
tính trên các mảng đã giữ, không khớp lại lần nào. Đây là lý do notebook này chạy 6 phút chứ
không phải vài giờ.

In [ ]:
t0 = time.time()
rng = np.random.default_rng(SEED)
VI_TRI = np.arange(len(d))
nen = {}
for v in XE:
    # Giữ lại chỉ số vị trí chứ không giữ cả DataFrame. Hai mươi bản sao của bảng
    # phiên sạc sẽ ngốn gần một gigabyte bộ nhớ mà không dùng tới.
    i_src = VI_TRI[(d.vehicle != v).values]
    i_tgt = VI_TRI[(d.vehicle == v).values]
    idx = rng.permutation(len(i_src))
    n_cal = min(N_CAL_NGUON, int(.2 * len(i_src)))
    i_cal, i_tr = i_src[idx[:n_cal]], i_src[idx[n_cal:]]
    assert len(np.intersect1d(i_tr, i_cal)) == 0, 'huấn luyện và hiệu chuẩn chồng nhau'
    assert len(np.intersect1d(i_tr, i_tgt)) == 0, 'xe đích lọt vào tập huấn luyện'

    tr, cal, tgt = d.iloc[i_tr], d.iloc[i_cal], d.iloc[i_tgt]
    m = tao_xgb().fit(tr[FEATS].values, tr.capacity.values)
    p_cal = np.clip(m.predict(cal[FEATS].values), CAP_LO, CAP_HI)
    p_tgt = np.clip(m.predict(tgt[FEATS].values), CAP_LO, CAP_HI)
    nen[v] = dict(idx_tr=i_tr, idx_cal=i_cal, idx_tgt=i_tgt,
                  res_cal=cal.capacity.values - p_cal,
                  res_tgt=tgt.capacity.values - p_tgt,
                  p_tgt=p_tgt, y_tgt=tgt.capacity.values,
                  y_cal=cal.capacity.values)
print(f'{len(XE)} lượt, {time.time()-t0:.0f} s')

hang = []
for v in XE:
    z = nen[v]
    q = q_conformal(z['res_cal'])
    bp, w = bao_phu_tu_res(z['res_tgt'], q)
    hang.append({'xe': v, 'n_dich': len(z['y_tgt']),
                 'RMSE': rmse(z['y_tgt'], z['p_tgt']),
                 'bao_phu_chi_nguon': bp, 'do_rong': w})
nen_df = pd.DataFrame(hang)
nen_df.to_csv(RES / 'nen_giu_xe.csv', index=False)
print(f"\nbao phủ trung bình {nen_df.bao_phu_chi_nguon.mean():.3f}, "
      f"mức danh nghĩa {NOM:.2f}")
print(f"{int((nen_df.bao_phu_chi_nguon < NOM).sum())} trên {len(nen_df)} xe dưới mức danh nghĩa, "
      f"xe tệ nhất {nen_df.bao_phu_chi_nguon.min():.3f}")
print('\nĐây là kết quả của Notebook B, dựng lại để ba bước sau có cùng một mốc so.')

## Bước 3, hướng R1: Mondrian, hiệu chuẩn riêng theo nhóm con

Conformal chuẩn dùng **một** phân vị cho mọi phiên. Mondrian chia không gian thành các nhóm
$G_1,\dots,G_m$ rồi lấy phân vị riêng trong từng nhóm. Cam kết bao phủ khi đó đúng **trong
từng nhóm**, chứ không chỉ đúng trên trung bình, và đó là thứ đáng giá: trung bình 0,885 có thể
che giấu một nhóm chỉ 0,76.

**Ràng buộc bắt buộc**: nhóm phải xác định được chỉ từ đầu vào $x$. Ở đây nhóm được gom cụm
k-means trên bốn mô tả kiểu sử dụng của phiên sạc, **khớp trên dữ liệu nguồn**, rồi áp cho xe
đích như một hàm đã cố định.

Lưu ý điều không làm được: không thể chia nhóm theo xe, vì xe đích theo định nghĩa không có mặt
trong tập hiệu chuẩn. Đó chính là lý do bài toán khó.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

MO_TA = [c for c in ['soc_lo', 'soc_span', 'i_mean', 'tmax_mean'] if c in FEATS_ALL]
print('gom cụm trên:', MO_TA)
M_NHOM = 4

def gan_nhom(v):
    # Cụm được khớp trên dữ liệu NGUỒN rồi áp cho xe đích như một hàm đã cố định.
    # Không có chỗ nào đụng tới dung lượng thật, nên nhóm gán được lúc dự báo.
    z = nen[v]
    A = d.iloc[z['idx_cal']][MO_TA].values
    B = d.iloc[z['idx_tgt']][MO_TA].values
    sc = StandardScaler().fit(A)
    km = KMeans(n_clusters=M_NHOM, n_init=10, random_state=SEED).fit(sc.transform(A))
    return km.predict(sc.transform(A)), km.predict(sc.transform(B))

t0 = time.time()
mon = []
for v in XE:
    z = nen[v]
    g_cal, g_tgt = gan_nhom(v)
    q_chung = q_conformal(z['res_cal'])
    bp_c, w_c = bao_phu_tu_res(z['res_tgt'], q_chung)

    q_nhom = {g: q_conformal(z['res_cal'][g_cal == g]) for g in range(M_NHOM)}
    qv = np.array([q_nhom[g] for g in g_tgt])
    phu = np.abs(z['res_tgt']) <= qv
    bp_m, w_m = float(phu.mean()), float(2 * np.median(qv))
    xau_nhat = min((float(phu[g_tgt == g].mean()) for g in range(M_NHOM)
                    if (g_tgt == g).sum() >= 20), default=np.nan)
    xau_nhat_c = min((float((np.abs(z['res_tgt'][g_tgt == g]) <= q_chung).mean())
                      for g in range(M_NHOM) if (g_tgt == g).sum() >= 20), default=np.nan)
    mon.append({'xe': v, 'bao_phu_chung': bp_c, 'bao_phu_mondrian': bp_m,
                'nhom_xau_nhat_chung': xau_nhat_c, 'nhom_xau_nhat_mondrian': xau_nhat,
                'do_rong_chung': w_c, 'do_rong_mondrian': w_m})
mon = pd.DataFrame(mon)
mon.to_csv(RES / 'mondrian.csv', index=False)
print(f'{time.time()-t0:.0f} s')
print(mon.describe().loc[['mean', 'min']].round(3).to_string())

In [ ]:
from scipy import stats as st

def p_wilcoxon(a, b):
    # Wilcoxon ném lỗi khi mọi hiệu bằng 0. Trường hợp đó p = 1 là câu trả lời đúng.
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    ok = np.isfinite(a) & np.isfinite(b)
    if not ok.any() or np.allclose(a[ok], b[ok]): return 1.0
    return float(st.wilcoxon(a[ok], b[ok]).pvalue)

p_bp = p_wilcoxon(mon.bao_phu_mondrian, mon.bao_phu_chung)
p_xn = p_wilcoxon(mon.nhom_xau_nhat_mondrian, mon.nhom_xau_nhat_chung)
bang = pd.DataFrame({
    'Chỉ số': ['bao phủ trung bình', 'bao phủ nhóm con tệ nhất', 'độ rộng trung vị, Ah',
               'số xe dưới mức danh nghĩa'],
    'Hiệu chuẩn chung': [mon.bao_phu_chung.mean(), mon.nhom_xau_nhat_chung.mean(),
                         mon.do_rong_chung.mean(), int((mon.bao_phu_chung < NOM).sum())],
    'Mondrian theo nhóm': [mon.bao_phu_mondrian.mean(), mon.nhom_xau_nhat_mondrian.mean(),
                           mon.do_rong_mondrian.mean(), int((mon.bao_phu_mondrian < NOM).sum())]})
print(bang.round(3).to_string(index=False))
print(f'\nWilcoxon bao phủ trung bình: p = {p_bp:.4f}')
print(f'Wilcoxon bao phủ nhóm tệ nhất: p = {p_xn:.4f}')

fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.2))
ax[0].scatter(mon.bao_phu_chung, mon.bao_phu_mondrian, s=26, color=TEAL)
lim = [min(mon.bao_phu_chung.min(), mon.bao_phu_mondrian.min()) - .02, 1.0]
ax[0].plot(lim, lim, color=SLATE, lw=.8, ls='--')
ax[0].axhline(NOM, color=RUST, lw=.8); ax[0].axvline(NOM, color=RUST, lw=.8)
ax[0].set_xlabel('bao phủ, hiệu chuẩn chung'); ax[0].set_ylabel('bao phủ, Mondrian')
ax[0].set_title('Mỗi điểm là một xe đích')
ax[1].scatter(mon.nhom_xau_nhat_chung, mon.nhom_xau_nhat_mondrian, s=26, color=RUST)
lim2 = [min(mon.nhom_xau_nhat_chung.min(), mon.nhom_xau_nhat_mondrian.min()) - .02, 1.0]
ax[1].plot(lim2, lim2, color=SLATE, lw=.8, ls='--')
ax[1].axhline(NOM, color=RUST, lw=.8); ax[1].axvline(NOM, color=RUST, lw=.8)
ax[1].set_xlabel('nhóm con tệ nhất, hiệu chuẩn chung')
ax[1].set_ylabel('nhóm con tệ nhất, Mondrian')
ax[1].set_title('Chỗ Mondrian đáng ra phải thắng')
plt.tight_layout(); plt.savefig(FIG / 'D_mondrian.png', dpi=160)
plt.show()

### Đọc kết quả Bước 3 cho đúng

Nhìn vào hai hình, trả lời bằng lời:

- Nếu các điểm nằm **trên** đường chéo ở hình phải nhưng bám sát đường chéo ở hình trái, Mondrian
  không sửa được trung bình mà sửa được chỗ tệ nhất. Đó vẫn là cải thiện thật và đáng viết.
- Nếu cả hai hình đều bám đường chéo, kết luận là **Mondrian không giúp gì ở bài toán này**, và
  lý do đáng viết ra: dịch chuyển nằm ở chiếc xe, không nằm ở kiểu phiên sạc. Chia nhóm theo
  kiểu phiên sạc thì mỗi nhóm vẫn trộn đủ 19 xe nguồn, nên vẫn thiếu đúng thứ đang thiếu.
- Mondrian luôn **tốn** một thứ: mỗi nhóm chỉ còn một phần tư số phiên hiệu chuẩn, nên phân vị
  nhiễu hơn. Nếu bao phủ không tăng thì đây là cái giá trả không công.

## Bước 4, hướng R2: cùng một ngân sách, chia cho ai

Ngân sách là **tổng** số phiên có nhãn trên cả đội xe, $K$. Hai cách chia:

| Cách | Quy tắc |
|---|---|
| Chia đều | mỗi xe nhận $K/20$ phiên |
| Thích nghi | chặng một chia đều một nửa ngân sách để **thăm dò**, ước lượng mức hụt của từng xe, chặng hai dồn nửa còn lại vào các xe hụt nhiều nhất |

Mức hụt được đo bằng phân vị sai số của riêng xe đó trên các phiên thăm dò, so với phân vị của
tập hiệu chuẩn nguồn. Xe nào có phân vị riêng lớn hơn hẳn thì phân vị chung đang quá hẹp với nó.

**Đánh giá luôn thực hiện trên các phiên chưa dùng làm nhãn**, ở cả hai cách, nếu không thì
phép so sánh vô nghĩa.

In [ ]:
def phan_bo(K_tong, cach, g):
    # Trả về số phiên có nhãn cho từng xe, tổng đúng bằng K_tong.
    k_deu = K_tong // len(XE)
    if cach == 'chia đều':
        return {v: k_deu for v in XE}
    k0 = max(5, k0_mac_dinh(k_deu))
    con_lai = K_tong - k0 * len(XE)
    trong_so = {}
    for v in XE:
        z = nen[v]
        pilot = g.choice(len(z['res_tgt']), size=min(k0, len(z['res_tgt'])), replace=False)
        trong_so[v] = max(0.0, q_conformal(z['res_tgt'][pilot]) - q_conformal(z['res_cal']))
    tong = sum(trong_so.values())
    if tong <= 0:
        them = {v: con_lai // len(XE) for v in XE}
    else:
        them = {v: int(round(con_lai * trong_so[v] / tong)) for v in XE}
    return {v: k0 + them[v] for v in XE}

def k0_mac_dinh(k_deu): return k_deu // 2

def danh_gia(cap_k, g):
    bp, rong = [], []
    for v in XE:
        z = nen[v]; r = z['res_tgt']; k = min(cap_k[v], len(r) - 50)
        idx = g.permutation(len(r))
        cal, te = idx[:max(k, 1)], idx[max(k, 1):]
        q = q_conformal(r[cal])
        b, w = bao_phu_tu_res(r[te], q)
        bp.append(b); rong.append(w)
    return np.array(bp), np.array(rong)

t0 = time.time()
KS_TONG = [200, 400, 600, 1000] if not NHANH else [200, 600]
N_LAP = 20
ns = []
for K in KS_TONG:
    for cach in ['chia đều', 'thích nghi']:
        B, W, A = [], [], []
        for r_ in range(N_LAP):
            g = np.random.default_rng(SEED + 1000 * r_ + K)
            cap_k = phan_bo(K, cach, g)
            B_, W_ = danh_gia(cap_k, g)
            B.append(B_); W.append(W_); A.append([cap_k[v] for v in XE])
        B, W, A = np.array(B), np.array(W), np.array(A)
        ns.append({'K_tong': K, 'cach': cach, 'k_moi_xe_neu_chia_deu': K // len(XE),
                   'bao_phu_tb': B.mean(), 'bao_phu_xe_te_nhat': B.min(1).mean(),
                   'so_xe_duoi_muc': (B < NOM).sum(1).mean(), 'do_rong_tb': W.mean(),
                   'k_it_nhat': A.min(), 'k_nhieu_nhat': A.max()})
ns = pd.DataFrame(ns)
ns.to_csv(RES / 'ngan_sach_thich_nghi.csv', index=False)
print(f'{time.time()-t0:.0f} s')
print(ns.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.9))
for cach, mauc, kh in [('chia đều', SAND, 'o-'), ('thích nghi', TEAL, 's-')]:
    g_ = ns[ns.cach == cach]
    ax[0].plot(g_.K_tong, g_.bao_phu_tb, kh, color=mauc, label=cach)
    ax[1].plot(g_.K_tong, g_.bao_phu_xe_te_nhat, kh, color=mauc, label=cach)
    ax[2].plot(g_.K_tong, g_.so_xe_duoi_muc, kh, color=mauc, label=cach)
for a, t in zip(ax, ['bao phủ trung bình', 'bao phủ của xe tệ nhất',
                     'số xe dưới mức danh nghĩa']):
    a.set_xlabel('tổng ngân sách nhãn K, phiên'); a.set_title(t); a.legend(fontsize=8)
ax[0].axhline(NOM, color=RUST, lw=.8, ls='--')
ax[1].axhline(NOM, color=RUST, lw=.8, ls='--')
plt.tight_layout(); plt.savefig(FIG / 'D_ngan_sach.png', dpi=160)
plt.show()

piv = ns.pivot(index='K_tong', columns='cach', values='bao_phu_xe_te_nhat')
piv['chênh'] = piv['thích nghi'] - piv['chia đều']
print('Bao phủ của xe tệ nhất, cùng một tổng ngân sách:')
print(piv.round(4).to_string())
print('\nCột chênh dương nghĩa là dồn nhãn vào xe yếu có lợi hơn chia đều.')
print('\nSố nhãn mà một xe thực sự nhận được, ít nhất và nhiều nhất:')
print(ns[['K_tong', 'cach', 'k_it_nhat', 'k_nhieu_nhat']].to_string(index=False))
print('\nHai cột cuối là chìa khoá để đọc bảng trên. Xem phần diễn giải ngay dưới.')

### Đọc kết quả Bước 4 cho đúng: hai hiện tượng dễ hiểu nhầm

**Hiện tượng 1: bao phủ trung bình không tăng đều theo ngân sách.** Với $k$ nhỏ, mức phân vị
$\lceil (n+1)(1-\alpha)\rceil/n$ bị kẹp về 1,0, nên $\hat q$ chính là **sai số lớn nhất** trong
$k$ phiên đó. Khoảng khi ấy rất rộng và bao phủ trung bình bị đẩy lên cao một cách giả tạo, đi
kèm độ rộng lớn. Nhìn riêng cột bao phủ trung bình sẽ tưởng ngân sách nhỏ là đủ. Nhìn kèm cột
độ rộng và cột xe tệ nhất thì thấy ngay là không. Đây là lý do mọi bảng trong notebook này đều
phải báo cáo ba cột cùng lúc.

**Hiện tượng 2: phân bổ thích nghi có thể thua chia đều ở ngân sách nhỏ.** Nhìn hai cột
`k_it_nhat` và `k_nhieu_nhat` vừa in ra. Cách thích nghi dành một nửa ngân sách cho chặng thăm
dò, nên ở $K$ nhỏ mỗi xe chỉ có vài phiên để ước lượng mức hụt của nó. Ước lượng từ vài phiên là
nhiễu, nên việc xếp hạng xe gần như ngẫu nhiên, mà cái giá thì có thật: xe bị xếp nhầm chỉ còn
đúng phần thăm dò, ít hơn hẳn so với chia đều.

Ở ngân sách lớn, chặng thăm dò đủ dài để xếp hạng đúng, và lúc đó phân bổ thích nghi mới vượt
lên. Nói cách khác, kết quả của bước này **không phải** "thích nghi tốt hơn" hay "thích nghi tệ
hơn", mà là: **phân bổ thích nghi có một ngưỡng ngân sách tối thiểu, dưới ngưỡng đó nó phản tác
dụng**. Đó là một câu có nội dung, và là câu nên viết vào bài.

Nếu muốn làm sâu thêm, đây là chỗ để thử: đổi tỉ lệ ngân sách dành cho chặng thăm dò trong hàm
`k0_mac_dinh`, rồi vẽ ngưỡng đó theo tỉ lệ. Bước này chạy trong vài giây nên quét được thoải mái.

## Bước 5, hướng R3: một phiên có nhãn nên dùng để huấn luyện hay để hiệu chuẩn

Câu hỏi mà bài báo nói thẳng là để ngỏ. Cho $k$ phiên có nhãn của xe đích, ba cách dùng:

| Cách | Mô hình | Tập hiệu chuẩn |
|---|---|---|
| Chỉ hiệu chuẩn lại | giữ nguyên, khớp trên 19 xe nguồn | $k$ phiên của xe đích |
| Chỉ huấn luyện lại | khớp lại trên 19 xe nguồn **cộng** $k$ phiên | hiệu chuẩn nguồn như cũ |
| Cả hai | khớp lại trên 19 xe nguồn cộng $k$ phiên | chính $k$ phiên đó |

Cách thứ ba có một chỗ tinh tế: dùng cùng $k$ phiên vừa để huấn luyện vừa để hiệu chuẩn thì
**phá vỡ điều kiện tách rời** của conformal, nên bao phủ của nó không còn được bảo đảm. Nó được
đưa vào đúng để đo xem vi phạm đó tốn bao nhiêu, chứ không phải để khuyến nghị.

Đây là bước duy nhất phải khớp lại mô hình, nên nó chiếm phần lớn thời gian chạy.

In [ ]:
t0 = time.time()
KS = [10, 25, 50, 100]
XE_R3 = XE[:6] if NHANH else XE[::max(1, len(XE)//8)][:8]
print('chạy trên', len(XE_R3), 'xe:', XE_R3)

hl = []
for so_v, v in enumerate(XE_R3):
    z = nen[v]
    r_tgt = z['res_tgt']; y_tgt = z['y_tgt']
    X_tr = d.iloc[z['idx_tr']][FEATS].values
    y_tr = d.iloc[z['idx_tr']].capacity.values
    X_cal = d.iloc[z['idx_cal']][FEATS].values
    X_tgt = d.iloc[z['idx_tgt']][FEATS].values
    g = np.random.default_rng(SEED + 7919 * so_v)
    for k in KS:
        idx = g.permutation(len(y_tgt))
        i_k, i_te = idx[:k], idx[k:]
        y_te = y_tgt[i_te]

        # 1. chỉ hiệu chuẩn lại, mô hình cũ
        q = q_conformal(r_tgt[i_k])
        b1, w1 = bao_phu_tu_res(r_tgt[i_te], q)
        r1 = rmse(y_te, z['p_tgt'][i_te])

        # 2 và 3. khớp lại một lần, dùng cho cả hai cách
        m2 = tao_xgb().fit(np.vstack([X_tr, X_tgt[i_k]]),
                           np.concatenate([y_tr, y_tgt[i_k]]))
        p_cal2 = np.clip(m2.predict(X_cal), CAP_LO, CAP_HI)
        p_te2 = np.clip(m2.predict(X_tgt[i_te]), CAP_LO, CAP_HI)
        p_k2 = np.clip(m2.predict(X_tgt[i_k]), CAP_LO, CAP_HI)
        r2_ = rmse(y_te, p_te2)
        b2, w2 = bao_phu_tu_res(y_te - p_te2, q_conformal(z['y_cal'] - p_cal2))
        b3, w3 = bao_phu_tu_res(y_te - p_te2, q_conformal(y_tgt[i_k] - p_k2))

        for ten, b, w, r in [('Chỉ hiệu chuẩn lại', b1, w1, r1),
                             ('Chỉ huấn luyện lại', b2, w2, r2_),
                             ('Cả hai, đã phá tách rời', b3, w3, r2_)]:
            hl.append({'xe': v, 'k': k, 'cach': ten, 'bao_phu': b, 'do_rong': w, 'RMSE': r})
    print(f'  xong {v}, {time.time()-t0:.0f} s')
hl = pd.DataFrame(hl)
hl.to_csv(RES / 'huan_luyen_hay_hieu_chuan.csv', index=False)
print(f'{time.time()-t0:.0f} s')

In [ ]:
tt = hl.groupby(['cach', 'k']).agg(bao_phu=('bao_phu', 'mean'),
                                   xe_te_nhat=('bao_phu', 'min'),
                                   do_rong=('do_rong', 'mean'),
                                   RMSE=('RMSE', 'mean')).reset_index()
print(tt.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.9))
mau3 = {'Chỉ hiệu chuẩn lại': TEAL, 'Chỉ huấn luyện lại': RUST,
        'Cả hai, đã phá tách rời': SLATE}
for cach, g_ in tt.groupby('cach'):
    ax[0].plot(g_.k, g_.bao_phu, 'o-', color=mau3[cach], label=cach)
    ax[1].plot(g_.k, g_.RMSE, 'o-', color=mau3[cach], label=cach)
    ax[2].plot(g_.k, g_.do_rong, 'o-', color=mau3[cach], label=cach)
ax[0].axhline(NOM, color=SAND, lw=1, ls='--'); ax[0].set_title('bao phủ')
ax[1].set_title('RMSE trên xe đích, Ah'); ax[2].set_title('độ rộng khoảng, Ah')
for a in ax:
    a.set_xlabel('số phiên có nhãn của xe đích, k'); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG / 'D_huan_luyen_hay_hieu_chuan.png', dpi=160)
plt.show()

hc = tt[tt.cach == 'Chỉ hiệu chuẩn lại'].set_index('k')
ht = tt[tt.cach == 'Chỉ huấn luyện lại'].set_index('k')
print(f'\nTại k = {KS[-1]}:')
print(f'  hiệu chuẩn lại  : bao phủ {hc.bao_phu[KS[-1]]:.3f}, RMSE {hc.RMSE[KS[-1]]:.2f} Ah')
print(f'  huấn luyện lại  : bao phủ {ht.bao_phu[KS[-1]]:.3f}, RMSE {ht.RMSE[KS[-1]]:.2f} Ah')
n_tr = len(nen[XE[0]]['idx_tr'])
print('\nRMSE gần như không đổi là điều phải xảy ra: k phiên thêm vào một tập huấn luyện')
print(f'{n_tr:,} phiên thì chìm nghỉm. Bao phủ thì đổi hẳn, vì phân vị')
print('conformal chỉ đọc đúng k phiên đó. Một nhãn đáng giá hơn nhiều khi dùng để hiệu chuẩn.')

## Bước 6: gộp ba kết quả lại thành một câu

Ba bước vừa rồi lần lượt gỡ ba giả định của Notebook B. Bảng dưới là chỗ để viết câu trả lời
bằng số, không phải bằng cảm giác.

In [ ]:
def an_toan(x, mac_dinh=np.nan):
    try: return float(x)
    except Exception: return mac_dinh

k_cuoi = KS[-1]
tom = pd.DataFrame([
    {'Hướng': 'Mốc, Notebook B', 'Thay đổi gì': 'chỉ hiệu chuẩn nguồn, không nhãn đích',
     'Bao phủ trung bình': nen_df.bao_phu_chi_nguon.mean(),
     'Xe tệ nhất': nen_df.bao_phu_chi_nguon.min(),
     'Độ rộng, Ah': nen_df.do_rong.mean()},
    {'Hướng': 'R1 Mondrian', 'Thay đổi gì': f'phân vị riêng cho {M_NHOM} nhóm phiên sạc',
     'Bao phủ trung bình': mon.bao_phu_mondrian.mean(),
     'Xe tệ nhất': mon.bao_phu_mondrian.min(),
     'Độ rộng, Ah': mon.do_rong_mondrian.mean()},
    {'Hướng': 'R2 ngân sách thích nghi',
     'Thay đổi gì': f'K = {KS_TONG[-1]} nhãn, dồn vào xe hụt nhiều nhất',
     'Bao phủ trung bình': an_toan(ns[(ns.K_tong == KS_TONG[-1]) &
                                      (ns.cach == 'thích nghi')].bao_phu_tb.iloc[0]),
     'Xe tệ nhất': an_toan(ns[(ns.K_tong == KS_TONG[-1]) &
                              (ns.cach == 'thích nghi')].bao_phu_xe_te_nhat.iloc[0]),
     'Độ rộng, Ah': an_toan(ns[(ns.K_tong == KS_TONG[-1]) &
                               (ns.cach == 'thích nghi')].do_rong_tb.iloc[0])},
    {'Hướng': 'R2 chia đều, để so',
     'Thay đổi gì': f'K = {KS_TONG[-1]} nhãn, chia đều cho 20 xe',
     'Bao phủ trung bình': an_toan(ns[(ns.K_tong == KS_TONG[-1]) &
                                      (ns.cach == 'chia đều')].bao_phu_tb.iloc[0]),
     'Xe tệ nhất': an_toan(ns[(ns.K_tong == KS_TONG[-1]) &
                              (ns.cach == 'chia đều')].bao_phu_xe_te_nhat.iloc[0]),
     'Độ rộng, Ah': an_toan(ns[(ns.K_tong == KS_TONG[-1]) &
                               (ns.cach == 'chia đều')].do_rong_tb.iloc[0])},
    {'Hướng': 'R3 huấn luyện lại', 'Thay đổi gì': f'{k_cuoi} nhãn đích đưa vào tập huấn luyện',
     'Bao phủ trung bình': float(ht.bao_phu[k_cuoi]), 'Xe tệ nhất': float(ht.xe_te_nhat[k_cuoi]),
     'Độ rộng, Ah': float(ht.do_rong[k_cuoi])},
    {'Hướng': 'R3 hiệu chuẩn lại', 'Thay đổi gì': f'{k_cuoi} nhãn đích dùng làm tập hiệu chuẩn',
     'Bao phủ trung bình': float(hc.bao_phu[k_cuoi]), 'Xe tệ nhất': float(hc.xe_te_nhat[k_cuoi]),
     'Độ rộng, Ah': float(hc.do_rong[k_cuoi])},
])
tom.to_csv(RES / 'tom_tat_D.csv', index=False)
print(tom.round(3).to_string(index=False))
print('\nMức danh nghĩa là 0,90. Cột "Xe tệ nhất" là cột quyết định, không phải cột trung bình:')
print('một phương pháp cho trung bình 0,90 mà có xe ở 0,76 thì chưa giữ được cam kết.')
print('\nĐọc kèm cột độ rộng. Bao phủ mua được bằng cách nới rộng khoảng thì không đáng gì:')
print('một khoảng rộng vô hạn phủ 100 % mà không nói lên điều gì. Phương pháp đáng giá là')
print('phương pháp nâng cột "Xe tệ nhất" mà **không** làm cột độ rộng phình theo.')

In [ ]:
xuat = {
    'nen_giu_xe.csv': 'mốc so, bao phủ khi chỉ hiệu chuẩn bằng dữ liệu nguồn',
    'mondrian.csv': 'R1, hiệu chuẩn chung so với hiệu chuẩn theo nhóm con',
    'ngan_sach_thich_nghi.csv': 'R2, chia đều so với dồn nhãn vào xe yếu',
    'huan_luyen_hay_hieu_chuan.csv': 'R3, ba cách dùng một phiên có nhãn',
    'tom_tat_D.csv': 'bảng gộp, đưa thẳng vào bài báo',
}
for f, mo_ta in xuat.items():
    print(f'{"CÓ " if (RES / f).exists() else "THIẾU"} {f:30s} {mo_ta}')
print('\nHình:', ', '.join(sorted(p.name for p in FIG.glob('D_*.png'))))

## Checklist trước khi nộp

| Mục | Đạt chưa |
|---|---|
| Nhóm Mondrian được gán **chỉ từ đầu vào**, không dùng dung lượng thật | |
| `assert` tập huấn luyện và tập hiệu chuẩn rời nhau đã chạy ở Bước 2 | |
| Ngân sách thích nghi tách hai chặng, chặng thăm dò không dùng để đánh giá | |
| Hai cách chia ngân sách được so ở **cùng một tổng K**, không phải cùng k mỗi xe | |
| Cách "Cả hai" được báo cáo kèm ghi chú đã phá điều kiện tách rời | |
| Mọi bảng báo cáo cả trung bình lẫn xe tệ nhất | |
| Kết quả âm được giữ nguyên, không chỉnh thí nghiệm cho ra số đẹp | |
| Audit Log có ít nhất 3 dòng trong tuần | |

## Việc tiếp theo của Sinh viên B

1. Viết Mục 6 (Discussion) của bài báo từ ba kết quả này. Cấu trúc gợi ý: bài báo chứng minh
   nhãn của xe đích là thứ không thể thay thế; notebook này cho thấy **cách chi nhãn** cũng có
   hạng, và hiệu chuẩn lại thắng huấn luyện lại ở mức chênh lệch lớn.
2. Chuẩn bị trả lời phản biện: *"vì sao không làm cả hai, vừa huấn luyện lại vừa hiệu chuẩn
   lại?"*. Số liệu ở Bước 5 dòng thứ ba đã có sẵn, kèm lý do vì sao cam kết mất hiệu lực.
3. Nếu còn thời gian, đây là chỗ mở sang hướng R4: dự báo dung lượng ở tương lai thay vì hiện
   tại. Toàn bộ khung ở Bước 2 dùng lại được, chỉ đổi cách chia tập theo mốc lịch.